In [1]:
import pandas as pd
import numpy as np
import datetime
import json
import os
from IPython.display import display_html
from IPython.display import HTML

In [2]:
def display_horizontal(*args):
    html_str = ''
    for df in args:
        html_str += df.to_html()
    display_html(
        html_str.replace('table','table style="display:inline;margin-right:50px"'), 
        raw=True
    )

In [3]:
with open("INDEX_START.json", "r", encoding="utf-8") as f:
    INDEX_START = json.load(f)
    
with open("INDEX_NAME.json", 'r', encoding = 'utf-8') as f:
    INDEX_NAME = json.load(f)

In [13]:
INDEX_DATA = {}

for ticker in INDEX_START:
    path = f"Data/{ticker}.csv"
    
    if os.path.isfile(path): 
        data = pd.read_csv(path, index_col = 0, parse_dates = True)
        data.index = pd.to_datetime(data.index, format="%Y-%m-%d", errors='coerce')
        data.index = data.index.date
        INDEX_DATA[ticker] = data.copy(deep = True)

In [14]:
UNIVERSE = {}
target_dates = {}
today_date = datetime.datetime(2025, 2, 21)

for ticker in INDEX_DATA:
    target_dates[ticker] = INDEX_DATA[ticker].index[-1]

UNIVERSE = INDEX_DATA

In [15]:
print(f"Assets under Analysis: {len(UNIVERSE)}")
print(f"Total Assets: {len(INDEX_DATA)}")

Assets under Analysis: 69
Total Assets: 69


## Z Score Calculation

In [16]:
Z_PE_TTM = {}
Z_DIVIDENDYIELD = {}
years = 5
duration = years * 250

for ticker in UNIVERSE:
    target_date = target_dates[ticker]
    if len(UNIVERSE[ticker]) < duration:
        continue
    
    # PE TTM 分位值
    sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std
    
    Z_PE_TTM[ticker] = Z
    
    
    # 股息率 分位值
    sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
    assert len(sample_data) == duration
    
    mean = sample_data.mean()
    std = sample_data.std()
    
    Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
    Z_DIVIDENDYIELD[ticker] = Z

In [17]:
display_number = 20

df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', '市盈率'])
df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', '股息率'])

# Sort the dataframes
df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='市盈率', ascending=True)
df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='股息率', ascending=False)

# Display the top 10 sorted rows
top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "市盈率"]]

top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "股息率"]]

In [18]:
print()
print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
print(f"{years}年")
display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
print()


2025-04-01
5年


,Ticker,指数名称,市盈率
15,931787CNY00.CSI,港股创新药,-1.247287
16,H30533.CSI,中概互联网,-1.190553
17,HSIII.HI,恒生互联网,-1.101265
31,707717L.MI,MSCI中国A股质量,-1.033177
2,000913.SH,医药,-0.926117
11,601688.SH,华泰证券,-0.817824
35,HSCGSI.HI,恒生消费,-0.773177
3,399814.SZ,农业,-0.746572
41,HSHK35.HI,恒生香港35,-0.709653
29,HSSCHI.HI,恒生港股通医疗,-0.651907


In [20]:
durations = [3, 5]

for years in durations:
    Z_PE_TTM = {}
    Z_DIVIDENDYIELD = {}
    target_date = INDEX_DATA['000300.SH'].index[-1]
    duration = years * 250

    for ticker in UNIVERSE:
        if len(UNIVERSE[ticker]) < duration:
            continue
    
        sample_data = UNIVERSE[ticker]['PE_TTM'].loc[:target_date].iloc[-duration:]
        assert len(sample_data) == duration
    
        mean = sample_data.mean()
        std = sample_data.std()
    
        Z = (UNIVERSE[ticker].loc[target_date, "PE_TTM"] - mean) / std

        if std == 0:
            print(ticker, duration, len(sample_data))
    
        Z_PE_TTM[ticker] = Z
    
    
        sample_data = UNIVERSE[ticker]['DIVIDENDYIELD2'].loc[:target_date].iloc[-duration:]
        assert len(sample_data) == duration
    
        mean = sample_data.mean()
        std = sample_data.std()
    
        Z = (UNIVERSE[ticker].loc[target_date, 'DIVIDENDYIELD2'] - mean) / std
    
        Z_DIVIDENDYIELD[ticker] = Z


    display_number = 20

    df_Z_PE_TTM = pd.DataFrame(list(Z_PE_TTM.items()), columns=['Ticker', '市盈率'])
    df_Z_DIVIDENDYIELD = pd.DataFrame(list(Z_DIVIDENDYIELD.items()), columns=['Ticker', '股息率'])

    # Sort the dataframes
    df_Z_PE_TTM_sorted = df_Z_PE_TTM.sort_values(by='市盈率', ascending=True)
    df_Z_DIVIDENDYIELD_sorted = df_Z_DIVIDENDYIELD.sort_values(by='股息率', ascending=False)

    # Display the top 10 sorted rows
    top_Z_PE_TTM = df_Z_PE_TTM_sorted.head(display_number).copy()
    top_Z_DIVIDENDYIELD = df_Z_DIVIDENDYIELD_sorted.head(display_number).copy()

    top_Z_PE_TTM.loc[:, "指数名称"] = top_Z_PE_TTM["Ticker"].map(INDEX_NAME)
    top_Z_PE_TTM = top_Z_PE_TTM[["Ticker", "指数名称", "市盈率"]]

    top_Z_DIVIDENDYIELD.loc[:, "指数名称"] = top_Z_DIVIDENDYIELD["Ticker"].map(INDEX_NAME)
    top_Z_DIVIDENDYIELD = top_Z_DIVIDENDYIELD[["Ticker", "指数名称", "股息率"]]


    print()
    print(f"{INDEX_DATA['000300.SH'].index[-1].strftime('%Y-%m-%d')}")
    print(f"{years}年")
    display_horizontal(top_Z_PE_TTM, top_Z_DIVIDENDYIELD)
    print()

KeyError: datetime.date(2025, 4, 1)

In [111]:
target_dates

{'NDX.GI': Timestamp('2025-02-21 00:00:00'),
 'SPX.GI': Timestamp('2025-02-21 00:00:00'),
 '000913.SH': Timestamp('2025-02-21 00:00:00'),
 '399814.SZ': Timestamp('2025-02-21 00:00:00'),
 '000300.SH': Timestamp('2025-02-21 00:00:00'),
 'HSI.HI': Timestamp('2025-02-21 00:00:00'),
 'HSTECH.HI': Timestamp('2025-02-21 00:00:00'),
 'NDXTMC.GI': Timestamp('2025-02-21 00:00:00'),
 'GDAXI.GI': Timestamp('2025-02-19 00:00:00'),
 'CSPSADRP.CI': Timestamp('2025-02-21 00:00:00'),
 '830009.XI': Timestamp('2025-02-21 00:00:00'),
 'N225.GI': Timestamp('2025-02-20 00:00:00'),
 '399975.SZ': Timestamp('2025-02-21 00:00:00'),
 '601688.SH': Timestamp('2025-02-21 00:00:00'),
 'HSSCHKY.HI': Timestamp('2025-02-21 00:00:00'),
 'HXC.GI': Timestamp('2025-02-21 00:00:00'),
 'HSHDYI.HI': Timestamp('2025-02-21 00:00:00'),
 'FCHI.GI': Timestamp('2025-02-19 00:00:00'),
 '931787CNY00.CSI': Timestamp('2025-02-21 00:00:00'),
 'H30533.CSI': Timestamp('2025-02-21 00:00:00'),
 'HSIII.HI': Timestamp('2025-02-21 00:00:00'),


22.19155531574851